In [0]:
%run /Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# ============================================================
# Data Quality Framework — tpch lakehouse
# Runs after gold layer build; fails pipeline on critical violations
# ============================================================

dq_results = []

def add_result(name, category, passed, failed_count, is_critical, details=""):
    """Record a DQ check result."""
    dq_results.append((name, category, passed, failed_count, is_critical, details))
    status = "✅" if passed else "❌"
    crit = " [CRITICAL]" if is_critical else " [INFO]"
    print(f"  {status} {name}: {failed_count:,} failures{crit}")
    if details and not passed:
        print(f"     → {details}")

source_tables = ["region", "nation", "customer", "supplier", "part", "partsupp", "orders", "lineitem"]

# ============================================================
# 1. Source Table Ingestion — all 8 tables in bronze, silver, gold
# ============================================================
print("=" * 70)
print("1. SOURCE TABLE INGESTION")
print("=" * 70)

for layer in ["bronze", "silver"]:
    for tbl in source_tables:
        count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.{layer}.{tbl}").collect()[0][0]
        passed = count > 0
        add_result(f"{layer}.{tbl} has data", "ingestion", passed, 0 if passed else 1, True,
                   f"rows: {count:,}" if count == 0 else "")

gold_tables = ["dim_geography", "dim_customer", "dim_product", "dim_supplier", "fact_sales"]
for tbl in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.gold.{tbl}").collect()[0][0]
    passed = count > 0
    add_result(f"gold.{tbl} has data", "ingestion", passed, 0 if passed else 1, True,
               f"rows: {count:,}" if count == 0 else "")

# ============================================================
# 2. Primary Key Integrity — populated and unique
# ============================================================
print("\n" + "=" * 70)
print("2. PRIMARY KEY INTEGRITY (silver layer)")
print("=" * 70)

pk_checks = [
    ("region", ["r_regionkey"]),
    ("nation", ["n_nationkey"]),
    ("customer", ["c_custkey"]),
    ("supplier", ["s_suppkey"]),
    ("part", ["p_partkey"]),
    ("orders", ["o_orderkey"]),
    ("partsupp", ["ps_partkey", "ps_suppkey"]),
    ("lineitem", ["l_orderkey", "l_linenumber"]),
]

for tbl, cols in pk_checks:
    cols_str = ", ".join(cols)
    null_cond = " OR ".join([f"{c} IS NULL" for c in cols])
    null_count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.silver.{tbl} WHERE {null_cond}").collect()[0][0]
    dup_count = spark.sql(f"""
        SELECT COALESCE(SUM(cnt - 1), 0) FROM (
            SELECT {cols_str}, COUNT(*) AS cnt
            FROM {catalog}.silver.{tbl}
            GROUP BY {cols_str}
            HAVING COUNT(*) > 1
        )
    """).collect()[0][0]

    total_fail = null_count + dup_count
    passed = total_fail == 0
    details = []
    if null_count > 0: details.append(f"nulls: {null_count:,}")
    if dup_count > 0: details.append(f"dups: {dup_count:,}")
    add_result(f"silver.{tbl} PK ({cols_str})", "primary_key", passed, total_fail, True,
               ", ".join(details) if details else "")

# ============================================================
# 3. Fact Table Surrogate Keys — no nulls
# ============================================================
print("\n" + "=" * 70)
print("3. FACT TABLE SURROGATE KEYS")
print("=" * 70)

fk_cols = ["order_key", "line_number", "customer_key", "product_key", "supplier_key"]
for col_name in fk_cols:
    null_count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.gold.fact_sales WHERE {col_name} IS NULL").collect()[0][0]
    add_result(f"fact_sales.{col_name} not null", "surrogate_key", null_count == 0, null_count, True)

# ============================================================
# 4. Numeric Measure Constraints
# ============================================================
print("\n" + "=" * 70)
print("4. NUMERIC MEASURE CONSTRAINTS")
print("=" * 70)

measure_checks = [
    (f"{catalog}.gold.fact_sales",    "discount in [0%, 10%]",        "discount >= 0 AND discount <= 0.10"),
    (f"{catalog}.gold.fact_sales",    "tax in [0%, 8%]",              "tax >= 0 AND tax <= 0.08"),
    (f"{catalog}.gold.fact_sales",    "extended_price > 0",          "extended_price > 0"),
    (f"{catalog}.gold.fact_sales",    "quantity > 0",                "quantity > 0"),
    (f"{catalog}.gold.fact_sales",    "net_price > 0",               "net_price > 0"),
    (f"{catalog}.gold.fact_sales",    "net_revenue > 0",             "net_revenue > 0"),
    (f"{catalog}.gold.fact_sales",    "order_total_price > 0",       "order_total_price > 0"),
    (f"{catalog}.gold.fact_sales",    "order_date in [1992, 1998]",  "order_date >= '1992-01-01' AND order_date <= '1998-12-31'"),
    (f"{catalog}.gold.fact_sales",    "ship_date in [1992, 1998]",   "ship_date >= '1992-01-01' AND ship_date <= '1998-12-31'"),
    (f"{catalog}.gold.fact_sales",    "commit_date in [1992, 1998]", "commit_date >= '1992-01-01' AND commit_date <= '1998-12-31'"),
    (f"{catalog}.gold.fact_sales",    "receipt_date in [1992, 1998]","receipt_date >= '1992-01-01' AND receipt_date <= '1998-12-31'"),
    (f"{catalog}.silver.lineitem",   "discount in [0%, 10%]",       "l_discount >= 0 AND l_discount <= 0.10"),
    (f"{catalog}.silver.lineitem",   "tax in [0%, 8%]",             "l_tax >= 0 AND l_tax <= 0.08"),
    (f"{catalog}.silver.lineitem",   "quantity > 0",                 "l_quantity > 0"),
    (f"{catalog}.silver.lineitem",   "extended_price > 0",          "l_extendedprice > 0"),
    (f"{catalog}.silver.orders",    "totalprice > 0",              "o_totalprice > 0"),
    (f"{catalog}.silver.part",       "size > 0",                     "p_size > 0"),
    (f"{catalog}.silver.part",       "retailprice > 0",             "p_retailprice > 0"),
    (f"{catalog}.silver.partsupp",   "availqty > 0",                 "ps_availqty > 0"),
    (f"{catalog}.silver.partsupp",   "supplycost > 0",               "ps_supplycost > 0"),
]

for full_tbl, desc, condition in measure_checks:
    fail_count = spark.sql(f"SELECT COUNT(*) FROM {full_tbl} WHERE NOT ({condition})").collect()[0][0]
    short_name = full_tbl.split(".")[-1]
    add_result(f"{short_name}: {desc}", "measure_constraint", fail_count == 0, fail_count, True)

# ============================================================
# 5. Gold-Silver Reconciliation
# ============================================================
print("\n" + "=" * 70)
print("5. GOLD-SILVER RECONCILIATION")
print("=" * 70)

# 5a. Row count: fact_sales == silver.lineitem
gold_count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.gold.fact_sales").collect()[0][0]
silver_count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.silver.lineitem").collect()[0][0]
diff = abs(gold_count - silver_count)
add_result("fact_sales rows == silver.lineitem rows", "reconciliation", diff == 0, diff, True,
           f"gold: {gold_count:,}, silver: {silver_count:,}")

# 5b. Net revenue: SUM(fact_sales.net_price) == SUM(silver.lineitem calc)
gold_rev = spark.sql(f"SELECT SUM(net_price) FROM {catalog}.gold.fact_sales").collect()[0][0]
silver_rev = spark.sql(f"SELECT SUM(l_extendedprice * (1 - l_discount)) FROM {catalog}.silver.lineitem").collect()[0][0]
rev_diff = abs(float(gold_rev) - float(silver_rev)) if gold_rev and silver_rev else float('inf')
add_result("fact_sales net_revenue == silver.lineitem net_revenue", "reconciliation",
           rev_diff < 0.01, int(rev_diff > 0.01), True,
           f"gold: {float(gold_rev):,.2f}, silver: {float(silver_rev):,.2f}, diff: {rev_diff:.4f}")

# 5c. Order count: distinct orders in fact_sales == silver.orders count
gold_orders = spark.sql(f"SELECT COUNT(DISTINCT order_key) FROM {catalog}.gold.fact_sales").collect()[0][0]
silver_orders = spark.sql(f"SELECT COUNT(*) FROM {catalog}.silver.orders").collect()[0][0]
order_diff = abs(gold_orders - silver_orders)
add_result("fact_sales distinct orders == silver.orders count", "reconciliation",
           order_diff == 0, order_diff, True,
           f"gold: {gold_orders:,}, silver: {silver_orders:,}")

# 5d. Customer FK: all customer_keys in fact_sales must exist in silver.customer
orphan_custs = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.gold.fact_sales f
    LEFT ANTI JOIN {catalog}.silver.customer c
        ON f.customer_key = c.c_custkey
""").collect()[0][0]
add_result("fact_sales customer_key FK to silver.customer", "reconciliation",
           orphan_custs == 0, orphan_custs, True,
           f"orphan customers: {orphan_custs:,}")

# 5e. Product FK: all product_keys in fact_sales must exist in silver.part
orphan_parts = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.gold.fact_sales f
    LEFT ANTI JOIN {catalog}.silver.part p
        ON f.product_key = p.p_partkey
""").collect()[0][0]
add_result("fact_sales product_key FK to silver.part", "reconciliation",
           orphan_parts == 0, orphan_parts, True,
           f"orphan products: {orphan_parts:,}")

# 5f. Supplier FK: all supplier_keys in fact_sales must exist in silver.supplier
orphan_supps = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.gold.fact_sales f
    LEFT ANTI JOIN {catalog}.silver.supplier s
        ON f.supplier_key = s.s_suppkey
""").collect()[0][0]
add_result("fact_sales supplier_key FK to silver.supplier", "reconciliation",
           orphan_supps == 0, orphan_supps, True,
           f"orphan suppliers: {orphan_supps:,}")

# ============================================================
# 6. Quarantine / Rejected Records (informational)
# ============================================================
print("\n" + "=" * 70)
print("6. QUARANTINE / REJECTED RECORDS")
print("=" * 70)

for tbl in source_tables:
    qt = f"{catalog}.silver.{tbl}_quarantine"
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM {qt}").collect()[0][0]
        add_result(f"silver.{tbl}_quarantine", "quarantine", True, count, False,
                   f"{count:,} rejected rows")
    except Exception:
        add_result(f"silver.{tbl}_quarantine", "quarantine", False, 1, False,
                   "table not found")

print("\n" + "=" * 70)
print("All DQ checks complete. See summary in next cell.")
print("=" * 70)

In [0]:
# ============================================================
# DQ Summary & Pipeline Gate
# ============================================================
# If any CRITICAL DQ check fails, this cell raises an Exception
# which terminates the notebook and causes the associated
# Databricks Job run to FAIL (non-zero exit).
# Non-critical (INFO) checks are reported but do not fail.
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, LongType

schema = StructType([
    StructField("check_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("passed", BooleanType(), True),
    StructField("failed_count", LongType(), True),
    StructField("is_critical", BooleanType(), True),
    StructField("details", StringType(), True),
])

results_df = spark.createDataFrame(dq_results, schema=schema)

print("=" * 70)
print("DATA QUALITY REPORT — SUMMARY")
print("=" * 70)
display(results_df.orderBy("category", "check_name"))

total = len(dq_results)
passed_count = sum(1 for r in dq_results if r[2])
failed_count = total - passed_count
critical_failures = sum(1 for r in dq_results if not r[2] and r[4])

print(f"\nTotal checks    : {total}")
print(f"Passed         : {passed_count}")
print(f"Failed         : {failed_count}")
print(f"Critical fails : {critical_failures}")

if critical_failures > 0:
    print(f"\n❌ PIPELINE FAILED: {critical_failures} critical DQ check(s) violated!")
    print("\nFailed critical checks:")
    for r in dq_results:
        if not r[2] and r[4]:
            print(f"   - {r[0]}: {r[5]}")
    print("\n>>> Job run terminated due to DQ critical failures. <<<")
    raise Exception(f"Data quality pipeline failed: {critical_failures} critical check(s) violated — job run terminated")
else:
    print(f"\n✅ All critical DQ checks passed — pipeline can proceed.")
    # Signal success to downstream job tasks
    dbutils.notebook.exit("DQ_PASSED")